In [ ]:
!pip install -q transformers datasets accelerate sentencepiece faiss-cpu sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.9 MB/s eta 0:00:00


<h2>Set Paths

In [ ]:
TRAIN_PATH = "/content/drive/MyDrive/Project/policy_data/train.json"
TEST_PATH  = "/content/drive/MyDrive/Project/policy_data/test.json"
MODEL_SAVE_PATH = "/content/drive/MyDrive/Project/bert-large"


<h2>Load & Prepare Dataset

In [ ]:
import json
from datasets import Dataset

def load_squad_like(path):
    with open(path, "r", encoding="utf-8") as f:
        js = json.load(f)

    examples = []
    for article in js["data"]:
        for para in article["paragraphs"]:
            context = para["context"]
            for qa in para["qas"]:
                answers = qa.get("answers", [])
                answers_dict = {
                    "text": [a["text"] for a in answers],
                    "answer_start": [a["answer_start"] for a in answers],
                }
                examples.append({
                    "id": qa.get("id", ""),
                    "context": context,
                    "question": qa["question"],
                    "answers": answers_dict
                })
    return Dataset.from_list(examples)

train_ds = load_squad_like(TRAIN_PATH)
val_ds = load_squad_like(TEST_PATH)

train_ds, val_ds


(Dataset({
     features: ['id', 'context', 'question', 'answers'],
     num_rows: 17056
 }),
 Dataset({
     features: ['id', 'context', 'question', 'answers'],
     num_rows: 4152
 }))

<h2> Tokenize Data

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-large-uncased-whole-word-masking"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 384
DOC_STRIDE = 128

def prepare_features(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LEN,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_map = tokenized.pop("offset_mapping")

    start_positions, end_positions = [], []

    for i, offsets in enumerate(offset_map):
        input_ids = tokenized["input_ids"][i]
        cls = input_ids.index(tokenizer.cls_token_id)

        sample_idx = sample_map[i]
        answers = examples["answers"][sample_idx]

        if len(answers["answer_start"]) == 0:
            start_positions.append(cls)
            end_positions.append(cls)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])
        sequence_ids = tokenized.sequence_ids(i)

        # Find context range
        context_start = next(i for i,s in enumerate(sequence_ids) if s == 1)
        context_end = len(sequence_ids) - 1 - next(i for i,s in enumerate(reversed(sequence_ids)) if s == 1)

        # Check if answer fits
        if not(offsets[context_start][0] <= start_char <= offsets[context_end][1]):
            start_positions.append(cls)
            end_positions.append(cls)
            continue

        # Map tokens
        start_token = context_start
        while start_token <= context_end and offsets[start_token][0] <= start_char:
            start_token += 1
        start_token -= 1

        end_token = context_end
        while end_token >= context_start and offsets[end_token][1] >= end_char:
            end_token -= 1
        end_token += 1

        start_positions.append(start_token)
        end_positions.append(end_token)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

tokenized_train = train_ds.map(
    prepare_features,
    batched=True,
    remove_columns=train_ds.column_names
)

tokenized_val = val_ds.map(
    prepare_features,
    batched=True,
    remove_columns=val_ds.column_names
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/17056 [00:00<?, ? examples/s]

Map:   0%|          | 0/4152 [00:00<?, ? examples/s]

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"


<h2>Train BERT-Large QA Model


In [ ]:
import torch
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME).to(device)

args = TrainingArguments(
    output_dir=MODEL_SAVE_PATH,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=100,
    save_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,  # BERT-large needs this
    learning_rate=3e-5,
    num_train_epochs=2,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val
)

trainer.train()

trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)


Using: cuda


model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-large-uncased-whole-word-masking
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

Step,Training Loss,Validation Loss
200,3.492810,3.484006
400,3.418196,3.423629
600,3.308133,3.340114
800,2.923505,2.864477
1000,2.615599,2.646060
1200,2.336251,2.590826
1400,2.176850,2.476469
1600,2.174877,2.449988
1800,2.100327,2.376022
2000,2.061618,2.359159


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Project/bert-large/tokenizer_config.json',
 '/content/drive/MyDrive/Project/bert-large/tokenizer.json')

In [ ]:
!pip install -q evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/84.1 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00


In [ ]:
import json
import torch
import numpy as np
from collections import defaultdict

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    default_data_collator,
)
import evaluate
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

MODEL_DIR = "/content/drive/MyDrive/Project/bert-large"  # <-- change if needed
TEST_PATH = "/content/drive/MyDrive/Project/policy_data/test.json"  # <-- change if needed


Device: cuda


In [ ]:
def load_squad_like(path):
    with open(path, "r", encoding="utf-8") as f:
        js = json.load(f)

    examples = []
    for article in js["data"]:
        for para in article["paragraphs"]:
            context = para["context"]
            for qa in para["qas"]:
                answers = qa.get("answers", [])
                answers_dict = {
                    "text": [a["text"] for a in answers],
                    "answer_start": [a["answer_start"] for a in answers],
                }
                examples.append(
                    {
                        "id": qa.get("id", ""),
                        "context": context,
                        "question": qa["question"],
                        "answers": answers_dict,
                    }
                )
    return Dataset.from_list(examples)

test_ds = load_squad_like(TEST_PATH)
print(test_ds[0])
print("Test examples:", len(test_ds))


{'id': '3f23wv3kh9cmvjio', 'context': 'Last Updated on May 22, 2015', 'question': "Do you take the user's opinion before or after making changes in policy?", 'answers': {'answer_start': [0], 'text': ['Last Updated on May 22, 2015']}}
Test examples: 4152


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForQuestionAnswering(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1024,), ep

In [ ]:
MAX_LEN = 384
DOC_STRIDE = 128

def prepare_test_features(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LEN,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []
    tokenized["offset_mapping"] = tokenized["offset_mapping"]

    for i in range(len(tokenized["input_ids"])):
        sample_idx = sample_mapping[i]
        tokenized["example_id"].append(examples["id"][sample_idx])

    return tokenized

test_features = test_ds.map(
    prepare_test_features,
    batched=True,
    remove_columns=test_ds.column_names,
)

print(test_features.column_names)
print("Features:", len(test_features))


Map:   0%|          | 0/4152 [00:00<?, ? examples/s]

['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id']
Features: 4164


In [ ]:
test_for_loader = test_features.remove_columns(["example_id", "offset_mapping"])

dataloader = DataLoader(
    test_for_loader,
    batch_size=8,
    shuffle=False,
    collate_fn=default_data_collator,
)

batch_example = next(iter(dataloader))
print(type(batch_example))
print(batch_example.keys())


<class 'dict'>
dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])


In [ ]:
all_start_logits = []
all_end_logits = []

for batch in tqdm(dataloader):
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    all_start_logits.append(outputs.start_logits.cpu().numpy())
    all_end_logits.append(outputs.end_logits.cpu().numpy())

all_start_logits = np.concatenate(all_start_logits, axis=0)
all_end_logits = np.concatenate(all_end_logits, axis=0)

print(all_start_logits.shape, all_end_logits.shape)


  0%|          | 0/521 [00:00<?, ?it/s]

(4164, 384) (4164, 384)


In [ ]:
metric = evaluate.load("squad")

example_id_to_index = {k: i for i, k in enumerate(test_ds["id"])}

features_per_example = defaultdict(list)
for i, feat_id in enumerate(test_features["example_id"]):
    features_per_example[feat_id].append(i)

max_answer_len = 30
predictions = []

for example in test_ds:
    example_id = example["id"]
    context = example["context"]
    feature_indices = features_per_example[example_id]

    best_answer = ""
    best_score = -1e9

    for idx in feature_indices:
        start_logits = all_start_logits[idx]
        end_logits = all_end_logits[idx]

        start_indexes = np.argsort(start_logits)[-5:][::-1]
        end_indexes = np.argsort(end_logits)[-5:][::-1]

        for s in start_indexes:
            for e in end_indexes:
                if e < s or e - s + 1 > max_answer_len:
                    continue
                score = start_logits[s] + end_logits[e]
                if score > best_score:
                    input_ids = test_features["input_ids"][idx][s:e+1]
                    text = tokenizer.decode(input_ids, skip_special_tokens=True)
                    best_answer = text
                    best_score = score

    predictions.append({"id": example_id, "prediction_text": best_answer})

references = [
    {"id": ex["id"], "answers": ex["answers"]} for ex in test_ds
]

results = metric.compute(predictions=predictions, references=references)
results


{'exact_match': 25.891136801541425, 'f1': 53.62791406371228}

In [ ]:
import torch

def ask_question(question, context):

    inputs = tokenizer(question, context, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    start_logits = outputs.start_logits
    end_logits = outputs.end_logits

    start = torch.argmax(start_logits)
    end = torch.argmax(end_logits)

    answer = tokenizer.decode(
        inputs["input_ids"][0][start:end+1],
        skip_special_tokens=True
    )

    # confidence score
    confidence = torch.softmax(start_logits, dim=-1)[0][start] * \
                 torch.softmax(end_logits, dim=-1)[0][end]

    confidence = float(confidence)

    return answer, confidence


In [ ]:
context = """
The device protection policy covers accidental damage such as screen cracks,
liquid spills, and internal hardware failures. A customer must report the
incident within 5 days of the damage occurring. To process the claim, the
customer must submit the purchase invoice, photos of the damage, and a valid
ID proof.

The policy provides one complimentary repair per year. If the repair cost
exceeds the insured amount, the remaining balance must be paid by the
customer. Damage caused by intentional misuse or unauthorized repairs is not
covered under this policy.

If the device cannot be repaired, the customer may be eligible for a
replacement unit, depending on stock availability and assessment results.
"""


In [ ]:
question = "Within how many days must the incident be reported?"
answer = ask_question(question, context)

print("Question:", question)
print("Answer:", answer)

Question: Within how many days must the incident be reported?
Answer: ('a customer must report the incident within 5 days of the damage occurring.', 0.33727535605430603)
